In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
from matplotlib.ticker import ScalarFormatter
from spectrum_component_analyser.spectral_component import spectral_component
from spectrum_component_analyser.mcmc.simulated_spectra import get_simulated_spectra
from spectrum_component_analyser.mcmc.constrained_helper import ConstrainedMCMCHelper
from phoenix_grid_creator.spectral_grid import spectral_grid

from constants import *
from astropy import units as u

from phoenix_grid_creator.spectral_grid import spectral_grid
from spectrum_component_analyser.phoenix_spectrum import phoenix_spectrum
from spectrum_component_analyser.chi_squared_minimisation import ChiHelper, get_chi_r

# both actual global constants
NUMBER_OF_PARAMETERS : int = 4 # weight, t, f, l

fits_file_paths = list(Path(package_path / "raw_phoenix_spectra").rglob("*.fits"))

In [ ]:
resolutions = np.logspace(np.log10(0.1), np.log10(0.0001), 6) * u.um
extra_resolutions = np.logspace(np.log10(0.001), np.log10(0.0001), 6) * u.um
resolutions = [*extra_resolutions, *resolutions]
resolutions = [0.01 * u.um]
resolutions = list(set(resolutions))
print(resolutions)

true_feh = 0.0 * u.dex
true_logg = 4.5 * u.dex

true_components = [
    spectral_component(3800 * u.K, true_feh, true_logg, 0.85),
    spectral_component(3300 * u.K, true_feh, true_logg, 0.15)
]

results = []

for res in resolutions:
    observational_wavelengths = np.arange(0.8, 5.5, res.value) * u.um
    temp_grid = spectral_grid.from_local_raw(fits_file_paths, False, observational_wavelengths)
    temp_grid.Wavelengths = temp_grid.Wavelengths.to(u.um)
    temp_grid.Fluxes *= u.Jy
    
    _, _, obs = get_simulated_spectra(temp_grid, true_components)
    
    chi_r = get_chi_r(len(true_components), NUMBER_OF_PARAMETERS, obs, temp_grid)
    
    # should be done by spectral grid class or the mcmchelper classes
    parameter_bounds = [
        (.0, 2.),
        (np.min(temp_grid.T_effs.value), np.max(temp_grid.T_effs.value)),
        (np.min(temp_grid.FeHs.value), np.max(temp_grid.FeHs.value)),
        (np.min(temp_grid.Log_gs.value), np.max(temp_grid.Log_gs.value)),
    ]

    mcmc = ConstrainedMCMCHelper(
        parameter_bounds=parameter_bounds,
        number_of_components=len(true_components),
        observed_spectrum=obs,
        spec_grid=temp_grid,
        n_steps=1500,
        n_walkers=int(2.5 * len(true_components) * NUMBER_OF_PARAMETERS)
    )
    
    sampler, samples = mcmc.run(chi_r)
    results.append(np.percentile(samples, [16, 50, 84], axis=0))

medians = np.array([r[1] for r in results])
lower_err = medians - np.array([r[0] for r in results])
upper_err = np.array([r[2] for r in results]) - medians

In [ ]:
# plot_corner(mcmc, sampler, samples, true_components)

%matplotlib inline
%config InlineBackend.figure_formats = ['svg']

In [ ]:
import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
fontsize=14
res_val = [v.value for v in resolutions]
comp_colors = ['royalblue', 'forestgreen']

for i in range(len(true_components)):
    t_true = true_components[i].T_eff.value
    w_true = true_components[i].Weight
    
    t_idx = i * 2 + 1
    w_idx = i * 2

    print(res)
    
    ax1.errorbar(res_val, medians[:, t_idx] - t_true, 
                 yerr=[lower_err[:, t_idx], upper_err[:, t_idx]],
                 fmt='o', color=comp_colors[i], capsize=5,
                 label=f'Component {i+1} ({t_true}K)')
    
    ax2.errorbar(res_val, medians[:, w_idx], 
                 yerr=[lower_err[:, w_idx], upper_err[:, w_idx]],
                 fmt='s', color=comp_colors[i], capsize=5)
                #  label=f'Component {i+1} (Truth: {w_true})')
    ax2.axhline(w_true, color=comp_colors[i], alpha=0.99, linestyle=':', label="$f_"+f"{i+1}"+f"={w_true}$ (ground truth)")

# real life resolutions
# ax1.axvline(2.2*u.um/150)
# ax1.axvline(0.8*u.um/150)
# ax1.axvline(5*u.um/1000)
# ax1.axvline(2*u.um/3500)
# ax1.axvline(0.8*u.um/150)


ax1.axhline(0, color='red', alpha=0.5)
ax1.set_ylabel(r"$T_{\mathrm{found}} - T_{\mathrm{true}}$ [K]",fontsize=fontsize+2)
ax1.legend(fontsize=fontsize, loc='lower right')
ax1.set_ylim(-850)

ax2.set_ylabel("$f$", fontsize=fontsize+2)
ax2.set_ylim(-0.08, 1.08)
ax2.set_xlabel(r"Resolution [$\mu$m]", fontsize=fontsize)
ax2.set_xscale('log')
# ax2.xaxis.set_major_formatter(ScalarFormatter())
leg = ax2.legend(fontsize=fontsize, loc='upper right')
for line in leg.get_lines():
    line.set_linewidth(2.0)
for ax in [ax1, ax2]:
    ax.grid(which='both', alpha=0.6)
    ax.invert_xaxis()
    ax.tick_params(axis='both', which='major', labelsize=12, length=8, width=1.5)

plt.tight_layout()
plt.savefig("high_res_plot.svg", format="svg", bbox_inches='tight')
plt.show()